# Intelligent Highway Lane Occupancy & Hogging Detection - Colab

This notebook is a thin wrapper around the `lane-analytics` CLI. The Python package is
not converted into notebook code.

The repository ships a **calibrated profile** for the Tj. Priok KM 58+400 (B) camera
(1024x768, night, 18 fps): six lanes across both carriageways, with the median-side lane
flagged as the overtaking lane in each direction.

Before running:

1. Select **Runtime > Change runtime type**. A GPU is optional; CPU works with `frame_stride`.
2. Clone the repository into `/content` (or upload a zip to Google Drive).
3. Put the highway clip in Google Drive, upload it, or record a public CCTV stream.
4. Run the cells from top to bottom.
5. Run `lane-analytics calibrate` once and check that the six polygons sit on the painted
   lines. Re-calibrate for any other camera.

The output is a rule-based **indication**, not a legal determination.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("GPU not available; CPU inference still works (use frame_stride 2-3).")


## 1. Get the project

Option A (recommended): clone the repository.

Option B: upload `highway-lane-analytics.zip` to `/content/drive/MyDrive/` and unzip it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess
import zipfile

PROJECT_DIR = "/content/highway-lane-analytics"
ZIP_PATH = "/content/drive/MyDrive/highway-lane-analytics.zip"
REPO_URL = "https://github.com/ibrahim119920/highway-lane-analytics.git"

if not os.path.isdir(PROJECT_DIR):
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH) as archive:
            archive.extractall("/content")
    else:
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

%cd {PROJECT_DIR}
!ls

## 2. Install dependencies

Colab already provides CUDA-enabled PyTorch. Install Ultralytics and the project without
replacing the existing torch build.


In [ ]:
!pip install -q ultralytics
!pip install -q -e . --no-deps
!lane-analytics doctor


## 3. Select the input video

Recommended: upload the clip to Google Drive and set `VIDEO_PATH`.


In [ ]:
import os

# The calibrated profile expects this clip. Upload it to Drive or use the upload cell below.
VIDEO_PATH = "/content/drive/MyDrive/km58_400B_tj_priok_20261107_161545.mp4"

# Alternative: upload a video from your computer.
# from google.colab import files
# uploaded = files.upload()
# VIDEO_PATH = "/content/" + next(iter(uploaded))

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
print("Using:", VIDEO_PATH)

## 3a. Optional: record a public toll-road CCTV stream

The Semarang-Solo toll-road cameras publish live HLS playlists. They are live 24/7
streams, not downloadable videos. `record_stream` records 30-second chunks, reconnects
when the playlist rotates, and remuxes the result.

Use this responsibly: only the currently broadcasting window is available, and repeated
aggressive requests should be avoided.


In [ ]:
from lane_analytics.recorder import record_stream

CCTV_CAMERA = "km486_500"   # "km437_800" or "km486_500"
RECORD_SECONDS = 300        # lower for a quick test

CCTV_STREAMS = {
    "km437_800": {
        "url": "https://jmlive.jasamarga.com/hls/15/f541b422-2339-4896-b329-ae65bcb3cef0/index.m3u8",
        "camera_id": "tmj_km437_800",
    },
    "km486_500": {
        "url": "https://jmlive.jasamarga.com/hls/15/c59b1716-ea0a-43ff-b9b0-d26bd863a428/index.m3u8",
        "camera_id": "tmj_km486_500",
    },
}

assert CCTV_CAMERA in CCTV_STREAMS
stream = CCTV_STREAMS[CCTV_CAMERA]
VIDEO_PATH = f"/content/cctv_{CCTV_CAMERA}.mp4"
CAMERA_ID = stream["camera_id"]

result = record_stream(
    stream["url"],
    VIDEO_PATH,
    duration=RECORD_SECONDS,
    chunk_seconds=30,
)
print(f"Requested {result.requested_seconds:.0f}s -> recorded {result.actual_seconds:.1f}s "
      f"({result.chunk_count} chunk(s), {result.attempts} attempt(s))")
print("Complete:", result.complete)
print("Recorded:", VIDEO_PATH)
print("camera_id:", CAMERA_ID)


## 4. Load the calibrated camera profile

The shipped `config/config.yaml` is already calibrated for this camera:

| lanes | role | notes |
| --- | --- | --- |
| `nb_lane_1`, `nb_lane_2` | normal | northbound outer and middle lanes |
| `nb_lane_3` | overtaking | northbound median-side lane |
| `sb_lane_1`, `sb_lane_2` | normal | southbound outer and middle lanes |
| `sb_lane_3` | overtaking | southbound median-side lane |

This cell only points the profile at your video and hardware. Do **not** overwrite the
polygons unless you are calibrating a different camera.

> **Observation window:** this camera sees a few tens of metres of roadway per direction,
> so a vehicle stays inside the lane polygons for roughly 0.6-2.5 s at highway speed.
> Hogging thresholds are set for that window (see the comments in `config/config.yaml`).

In [ ]:
import yaml

CONFIG_PATH = "config/config.yaml"
with open(CONFIG_PATH, encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

config["video"]["path"] = VIDEO_PATH
config["video"]["frame_stride"] = 2          # 1 on GPU, 2-3 on CPU
config["model"]["device"] = "0" if __import__("torch").cuda.is_available() else "cpu"
config["model"]["imgsz"] = 960               # drop to 640 on a slow CPU
config["camera_id"] = globals().get("CAMERA_ID", config.get("camera_id"))

with open(CONFIG_PATH, "w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, sort_keys=False, allow_unicode=True)

print("lanes :", [lane["name"] for lane in config["lanes"]])
print("video :", config["video"])
print("model :", config["model"])

### 4a. Calibration grid and polygon preview

`calibrate` renders one frame with a labelled pixel grid, a normalized y ruler, and the
configured polygons. Read a lane edge at two heights, divide `x` by the frame width and `y`
by the frame height, and edit `config/config.yaml` if the polygons do not sit on the paint.

`preview-lanes` renders the polygons without the grid.

In [ ]:
FRAME = 2160  # source frame index; 2160 ~= 120 s into the 18 fps clip

!lane-analytics calibrate --frame {FRAME} --out output/colab_calibration.jpg
!lane-analytics preview-lanes --frame {FRAME} --out output/colab_lane_preview.jpg

from IPython.display import Image, display
display(Image(filename="output/colab_lane_preview.jpg"))

## 5. Run YOLO + ByteTrack

For a quick smoke test, add `--max-frames 300` or `--no-video`.


In [ ]:
import subprocess

command = [
    "lane-analytics", "analyze",
    "--config", "config/config.yaml",
    "--device", config["model"]["device"],
    "--frame-stride", str(config["video"]["frame_stride"]),
]
print(" ".join(command))
subprocess.run(command, check=True)


## 6. Play the annotated result

Browsers do not reliably play OpenCV's `mp4v` codec. The file is re-encoded to H.264 for
inline playback. For long clips, use `embed=False` or download the file.


In [ ]:
from pathlib import Path
import subprocess

videos = sorted(
    Path("output/videos").glob("*_annotated.mp4"),
    key=lambda path: path.stat().st_mtime,
)
assert videos, "No annotated video found - run the analysis cell first."
video = videos[-1]
browser_video = video.with_name(video.stem + "_h264.mp4")

subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(video),
        "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(browser_video),
    ],
    check=True,
)

from IPython.display import Video
print(browser_video)
Video(str(browser_video), embed=True, width=960)


## 7. Review the lane occupancy and hogging report


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import pandas as pd

reports = sorted(
    Path("output/reports").glob("*/summary.md"),
    key=lambda path: path.stat().st_mtime,
)
assert reports, "No report found - run the analysis cell first."
report_dir = reports[-1].parent
display(Markdown(reports[-1].read_text(encoding="utf-8")))

for name in ("lane_usage.csv", "hogging_events.csv", "lane_changes.csv"):
    path = report_dir / name
    if path.exists():
        print(f"\n--- {name} ---")
        display(pd.read_csv(path).head(20))


## 8. Save results to Google Drive

Colab deletes `/content` when the runtime ends. Copy `output/` to Drive before closing.


In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

destination = (
    Path("/content/drive/MyDrive/lane-occupancy-output")
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)
shutil.copytree("output", destination, dirs_exist_ok=True)
print("Saved to", destination)


## Notes

- The offline synthetic demo also works on Colab (CPU is enough):

  ```python
  !lane-analytics demo --output-dir output/demo
  ```

- CPU inference: set `video.frame_stride` to 2-3 and `model.imgsz` to 640.
- Video decoding and OpenCV annotation still use CPU; `--no-video` maximizes data throughput.
- A track ID can switch after a long occlusion; review `lane_stints.csv` before drawing
  conclusions about a single vehicle.
- For this camera the lane polygons cover only part of the visible road, so `dwell_seconds`
  means "time observed inside the analysis zone". Raise the hogging thresholds if you
  calibrate a camera with a longer view.
- Both carriageways are mapped. `lane_index` is scoped per direction, so use `lane_name`
  when joining with downstream data.
- Lane-hogging rules are configurable heuristics. Review thresholds against the actual
  road rules before using the output for anything beyond research.
- Use footage you have the right to process. Ultralytics YOLO has its own license terms.